In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading & Preprocessing
We load the clean combined dataset which contains all the features we need.

In [8]:
df = pd.read_csv('data/truecar_clean_combined.csv')
print(f"Initial data shape: {df.shape}")

# Filter missing critical values
df = df.dropna(subset=['sales_price', 'year', 'make', 'model', 'odometer_miles', 'fuel_type'])

# Create the vehicle_age feature (assuming current year is 2026)
df['vehicle_age'] = 2026 - df['year']
df['vehicle_age'] = df['vehicle_age'].apply(lambda x: 0 if x < 0 else x)

print(f"Data shape after dropping NAs: {df.shape}")

Initial data shape: (8629, 46)
Data shape after dropping NAs: (8594, 46)


## 2. Model Training
We'll build a Random Forest Regressor to predict `sales_price` based on features like age, mileage, make, model, and fuel type.

In [9]:
features = ['vehicle_age', 'odometer_miles', 'make', 'model', 'fuel_type', 'feature_count']
target = 'sales_price'

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing for categorical data
categorical_features = ['make', 'model', 'fuel_type']
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ], remainder='passthrough')

# Create and train pipeline with Hyperparameter Tuning
print("Setting up hyperparameter tuning via RandomizedSearchCV...")

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_distributions = {
    'regressor__n_estimators': [50, 100, 200],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__bootstrap': [True, False],
    'regressor__max_features': ['sqrt', 'log2', None],
    'regressor__criterion': ['squared_error', 'absolute_error']
}

# Note: 'gini' and 'entropy' were excluded because they are classification metrics, and this is a regression problem (predicting price)
random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Training model with hyperparameter tuning...")
random_search.fit(X_train, y_train)

print(f"Best parameters found: {random_search.best_params_}")

# Update model to be the best estimator
model = random_search.best_estimator_

# Evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"\nMean Absolute Error on Test Set using best parameters: ${mae:,.2f}")


Setting up hyperparameter tuning via RandomizedSearchCV...
Training model with hyperparameter tuning...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters found: {'regressor__n_estimators': 200, 'regressor__max_features': 'sqrt', 'regressor__max_depth': 30, 'regressor__criterion': 'absolute_error', 'regressor__bootstrap': True}

Mean Absolute Error on Test Set using best parameters: $5,188.89


## 3. Recommendation Engine
This function filters the dataset based on user preferences and calculates a 'Discount' (Predicted Fair Price - Actual Price) to find the best deals.

In [10]:
def suggest_cars(df, model, max_price=None, max_miles_per_year=8000, 
                 require_hybrid=False, require_awd=False, 
                 require_sunroof=False, require_high_clearance=False):
    
    # Work on a copy
    recs = df.copy()
    
    # Predict prices for all cars in the dataset so we can find discounts
    recs['predicted_fair_price'] = model.predict(recs[['vehicle_age', 'odometer_miles', 'make', 'model', 'fuel_type', 'feature_count']])
    recs['discount'] = recs['predicted_fair_price'] - recs['sales_price']
    
    # 1. Budget Constraint
    if max_price:
        recs = recs[recs['sales_price'] <= max_price]
        
    # 2. Mileage Constraint (Max 8k miles per year)
    # If age is 0 (brand new), we allow up to max_miles_per_year to avoid division by zero
    recs['allowed_miles'] = recs['vehicle_age'].apply(lambda age: max_miles_per_year if age == 0 else age * max_miles_per_year)
    recs = recs[recs['odometer_miles'] <= recs['allowed_miles']]
    
    # 3. Hybrid / Plug-in
    if require_hybrid:
        recs = recs[recs['fuel_type'].str.contains('hybrid|plug-in', case=False, na=False)]
        
    # 4. AWD / 4WD
    if require_awd:
        # Check trim, model, or title for AWD/4WD/4x4 indications
        awd_mask = recs['title'].str.contains('AWD|4WD|4x4|All-Wheel|Four-Wheel', case=False, na=False) | \
                   recs['trim'].str.contains('AWD|4WD|4x4', case=False, na=False)
        recs = recs[awd_mask]
        
    # 5. Sunroof / Moonroof
    if require_sunroof:
        # Check across multiple text fields for sunroof/moonroof
        sunroof_mask = recs.apply(lambda row: row.astype(str).str.contains('sunroof|moonroof|panoramic', case=False).any(), axis=1)
        recs = recs[sunroof_mask]
        
    # 6. Higher Ground Clearance (Approximation via Body Style / Model)
    if require_high_clearance:
        # Since we don't have exact ground clearance, we filter for SUVs, Trucks, and Crossovers
        # Assuming we check the exterior, title, or model for SUV/Truck keywords
        high_clearance_keywords = 'SUV|Truck|Jeep|Rover|Outback|Crosstrek|Bronco|Wrangler|Highlander|RAV4|CR-V|Pilot|Explorer|Tahoe'
        hc_mask = recs.apply(lambda row: row.astype(str).str.contains(high_clearance_keywords, case=False).any(), axis=1)
        recs = recs[hc_mask]
        
    # Sort by the best "Discount" (Predicted - Actual)
    recs = recs.sort_values(by='discount', ascending=False)
    
    return recs[['year', 'make', 'model', 'trim', 'odometer_miles', 'fuel_type', 'sales_price', 'predicted_fair_price', 'discount', 'url']]


## 4. Run Recommendation based on Preferences
Here we test the recommendation engine using the specific constraints: max 8k miles/year, hybrid/plugin, AWD, high ground clearance, and sunroof.

In [11]:
print("Finding the perfect vehicles based on preferences...")
best_deals = suggest_cars(df, model, 
                          max_miles_per_year=8000,
                          require_hybrid=True,
                          require_awd=True,
                          require_sunroof=True,
                          require_high_clearance=True)

if not best_deals.empty:
    print(f"Found {len(best_deals)} matching cars!")
    display(best_deals.head(10).style.format({
        'sales_price': '${:,.0f}', 
        'predicted_fair_price': '${:,.0f}',
        'discount': '${:,.0f}',
        'odometer_miles': '{:,.0f}'
    }).background_gradient(subset=['discount'], cmap='Greens'))
else:
    print("No cars found matching all these strict constraints in the current dataset.")
    print("Try relaxing some constraints (e.g., allow higher mileage or remove sunroof requirement).")

Finding the perfect vehicles based on preferences...
Found 15 matching cars!


,year,make,model,trim,odometer_miles,fuel_type,sales_price,predicted_fair_price,discount,url
482,2022,Jeep,Grand,Cherokee 4xe 4WD,"19,214",Plug-In Hybrid,"$29,000","$30,767","$1,767",https://www.truecar.com/used-cars-for-sale/listing/1C4RJYB64N8756265/2022-jeep-grand-cherokee/?position=19&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-chicago-il%2F%3Fpage%3D6&sourceType=marketplace
481,2022,Jeep,Grand,Cherokee 4xe 4WD,"21,816",Plug-In Hybrid,"$28,999","$29,932",$933,https://www.truecar.com/used-cars-for-sale/listing/1C4RJYB63N8761716/2022-jeep-grand-cherokee/?position=13&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-chicago-il%2F%3Fpage%3D14&sourceType=marketplace
477,2022,Jeep,Grand,Cherokee 4xe 4WD,"18,056",Plug-In Hybrid,"$30,977","$31,372",$395,https://www.truecar.com/used-cars-for-sale/listing/1C4RJYB61N8757583/2022-jeep-grand-cherokee/?position=14&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-boston-ma%2F%3Fpage%3D28&sourceType=marketplace
478,2022,Jeep,Grand,Cherokee 4xe 4WD,"15,123",Plug-In Hybrid,"$31,900","$31,760",$-140,https://www.truecar.com/used-cars-for-sale/listing/1C4RJYB62N8724673/2022-jeep-grand-cherokee/?position=21&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-austin-tx%2F%3Fpage%3D30&sourceType=marketplace
485,2024,Jeep,Grand,Cherokee 4xe 4WD,"8,035",Plug-In Hybrid,"$38,384","$37,214","$-1,170",https://www.truecar.com/used-cars-for-sale/listing/1C4RJYB69RC107063/2024-jeep-grand-cherokee/?position=1&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-austin-tx%2F%3Fpage%3D6&sourceType=marketplace
3405,2024,Toyota,RAV4,Hybrid XLE AWD,"15,020",Hybrid,"$35,187","$33,104","$-2,083",https://www.truecar.com/used-cars-for-sale/listing/2T3RWRFV7RW208012/2024-toyota-rav4/?position=10&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-boston-ma%2F%3Fpage%3D46&sourceType=marketplace
8481,2019,Volvo,XC90,T8 Inscription Plug-In Hybrid eAWD,"49,502",Plug-In Hybrid,"$31,995","$29,779","$-2,216",https://www.truecar.com/used-cars-for-sale/listing/YV4BR0CLXK1439754/2019-volvo-xc90/?position=2&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-chicago-il%2F%3Fpage%3D17&sourceType=marketplace
5990,2020,Honda,CR-V,Hybrid Touring AWD,"35,526",Hybrid,"$29,145","$26,814","$-2,331",https://www.truecar.com/used-cars-for-sale/listing/7FART6H94LE000131/2020-honda-cr-v/?position=0&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-detroit-mi%2F%3Fpage%3D39&sourceType=marketplace
3404,2024,Toyota,RAV4,Hybrid XLE AWD,"15,663",Hybrid,"$35,702","$33,077","$-2,625",https://www.truecar.com/used-cars-for-sale/listing/2T3RWRFV2RW238762/2024-toyota-rav4/?position=0&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-austin-tx%2F%3Fpage%3D15&sourceType=marketplace
5985,2025,Honda,CR-V,Hybrid Sport AWD,"1,321",Hybrid,"$36,856","$33,890","$-2,966",https://www.truecar.com/used-cars-for-sale/listing/7FARS6H5XSE030152/2025-honda-cr-v/?position=17&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-boston-ma%2F%3Fpage%3D47&sourceType=marketplace


In [12]:
print("Finding the perfect vehicles based on preferences...")
best_deals = suggest_cars(df, model, 
                          max_miles_per_year=9000,
                          require_hybrid=True,
                          require_awd=True,
                          require_sunroof=False,
                          require_high_clearance=False)

if not best_deals.empty:
    print(f"Found {len(best_deals)} matching cars!")
    display(best_deals.head(10).style.format({
        'sales_price': '${:,.0f}', 
        'predicted_fair_price': '${:,.0f}',
        'discount': '${:,.0f}',
        'odometer_miles': '{:,.0f}'
    }).background_gradient(subset=['discount'], cmap='Greens'))
else:
    print("No cars found matching all these strict constraints in the current dataset.")
    print("Try relaxing some constraints (e.g., allow higher mileage or remove sunroof requirement).")

Finding the perfect vehicles based on preferences...
Found 71 matching cars!


,year,make,model,trim,odometer_miles,fuel_type,sales_price,predicted_fair_price,discount,url
8537,2024,Dodge,Hornet,R/T Plus EAWD,"6,849",Plug-In Hybrid,"$28,995","$38,771","$9,776",https://www.truecar.com/used-cars-for-sale/listing/ZACPDFDW0R3A14094/2024-dodge-hornet/?position=16&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-chicago-il%2F%3Fpage%3D35&sourceType=marketplace
8538,2024,Dodge,Hornet,R/T Plus EAWD,"1,527",Plug-In Hybrid,"$31,998","$39,209","$7,211",https://www.truecar.com/used-cars-for-sale/listing/ZACPDFDW2R3A24383/2024-dodge-hornet/?position=7&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-seattle-wa%2F%3Fpage%3D20&sourceType=marketplace&zipcode=98057
8563,2024,Alfa,Romeo,Tonale Ti EAWD,"4,726",Plug-In Hybrid,"$33,000","$38,800","$5,800",https://www.truecar.com/used-cars-for-sale/listing/ZASPATCW5R3055590/2024-alfa-romeo-tonale/?position=12&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-boston-ma%2F%3Fpage%3D50&sourceType=marketplace
6515,2024,Toyota,Corolla,Hybrid SE AWD,"1,590",Hybrid,"$26,500","$31,253","$4,753",https://www.truecar.com/used-cars-for-sale/listing/JTDBDMHE8R3010923/2024-toyota-corolla/?position=3&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-boston-ma%2F%3Fpage%3D28&sourceType=marketplace
5548,2021,BMW,X3,Plug-In Hybrid xDrive30e AWD,"24,803",Plug-In Hybrid,"$32,377","$35,833","$3,456",https://www.truecar.com/used-cars-for-sale/listing/5UXTS1C0XM9E84741/2021-bmw-x3/?position=12&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-boston-ma%2F%3Fpage%3D47&sourceType=marketplace
476,2023,Jeep,Grand,Cherokee 4xe 4WD,"8,327",Plug-In Hybrid,"$32,900","$35,844","$2,944",https://www.truecar.com/used-cars-for-sale/listing/1C4RJYB60P8774734/2023-jeep-grand-cherokee/?position=19&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-austin-tx%2F%3Fpage%3D6&sourceType=marketplace
943,2022,Ford,Escape,Titanium Hybrid AWD,"20,000",Hybrid,"$24,670","$27,012","$2,342",https://www.truecar.com/used-cars-for-sale/listing/1FMCU9DZ1NUA07968/2022-ford-escape/?position=7&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-san-francisco-ca%2F%3Fpage%3D48&sourceType=marketplace
482,2022,Jeep,Grand,Cherokee 4xe 4WD,"19,214",Plug-In Hybrid,"$29,000","$30,767","$1,767",https://www.truecar.com/used-cars-for-sale/listing/1C4RJYB64N8756265/2022-jeep-grand-cherokee/?position=19&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-chicago-il%2F%3Fpage%3D6&sourceType=marketplace
6606,2007,Toyota,Highlander,Hybrid 4WD,"145,387",Hybrid,"$9,985","$11,685","$1,700",https://www.truecar.com/used-cars-for-sale/listing/JTEHW21A970037042/2007-toyota-highlander/?position=19&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-seattle-wa%2F%3Fpage%3D38&sourceType=marketplace
6713,2008,Lexus,RX,400h Hybrid AWD,"113,141",Hybrid,"$10,500","$12,188","$1,688",https://www.truecar.com/used-cars-for-sale/listing/JTJHW31U682045803/2008-lexus-rx/?position=0&returnTo=%2Fused-cars-for-sale%2Flistings%2Flocation-san-francisco-ca%2F%3Fpage%3D11&sourceType=marketplace
